# ELUATE → OCCLUDE — Google Colab pipeline

Two-stage chain on a single GPU runtime:

1. **ELUATE** strips the background music, keeps speech + sfx, copies the video stream through untouched.
2. **Transcode to H.264** — OCCLUDE decodes frames via OpenCV, which can't read AV1 (common for Google/YouTube-sourced files). The system ffmpeg can; one transcode bridges the gap.
3. **OCCLUDE** blurs immodestly-dressed people frame-by-frame and remuxes the (already music-free) audio.

Both tools installed from PyPI — no git clone, no repo access needed.

**All compute is on CUDA** (occlude ≥ 1.1.2): YOLO detection (fp16), SegFormer segmentation (fp16 + `torch.compile`), its image preprocessing, the per-frame pixelate/Gaussian blur, InsightFace face/gender (CUDA ONNX EP), and CUDA video I/O when torchcodec/NVENC are available. Cell 6 prints a per-model device check and refuses to start the run if anything falls off CUDA.

**Runtime (Pro+)**: Runtime → Change runtime type → **A100** (fastest for both stages). Fall back to **L4** if A100 is unavailable. Enable **background execution** so the job survives a closed tab.

**First-batch warm-up**: in the **full run (cell 6)** `torch.compile` JIT-fuses the SegFormer kernels on the first perception batch — the first ~10–30 frames crawl, then accelerate. Expected, not a hang; the warm-up is negligible across a feature-length video. The short **validation (cell 8)** and **profile (cell 9)** cells deliberately set `TORCH_COMPILE_DISABLE=1` so a 30–90 s sample measures steady-state fps, not the one-time compile.

**onnxruntime note**: plain `onnxruntime` is an occlude core dependency and *shadows* `onnxruntime-gpu`, so `pip install 'occlude[gpu]'` **alone is not enough**. Every run cell (6, 8, 9) repairs this by uninstalling both and force-reinstalling `onnxruntime-gpu` last and alone; CUDA verification hard-stops if InsightFace cannot bind the CUDA provider. Don't remove that block.

**Drive layout** this notebook expects:
```
MyDrive/
  occlude/
    inputs/   <- put source videos here
    outputs/  <- final (music-removed + blurred) videos land here
    models/   <- (auto) cached weights: Bandit v2, SegFormer, InsightFace, YOLO
```

In [ ]:
# 1. Verify GPU (expect A100 / L4 on Pro+)
!nvidia-smi

In [ ]:
# 2. System deps: ffmpeg (AV1 decode via libdav1d + remux), libgl1
#    (opencv runtime).
!apt-get -qq install -y ffmpeg libgl1 > /dev/null

In [ ]:
# 3. Install both tools from PyPI. Their pyproject metadata pulls every
#    dependency; nothing else to install by hand.
!pip install -q eluate occlude

In [ ]:
# 4. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5. Persist model weights to Drive so later sessions skip the downloads.
import os
from pathlib import Path

DRIVE = "/content/drive/MyDrive/occlude"
os.makedirs(f"{DRIVE}/models", exist_ok=True)

# OCCLUDE caches (env-var driven, inherited by the subprocess in cell 6).
os.environ["HF_HOME"]          = f"{DRIVE}/models/hf"
os.environ["INSIGHTFACE_HOME"] = f"{DRIVE}/models/insightface"

# ELUATE's app dir is hardcoded to ~/.eluate; symlink it to Drive so the
# Bandit v2 checkpoint persists across sessions.
home_eluate  = Path.home() / ".eluate"
drive_eluate = Path(f"{DRIVE}/models/eluate")
drive_eluate.mkdir(parents=True, exist_ok=True)
if not home_eluate.exists():
    os.symlink(drive_eluate, home_eluate)

# YOLO downloads yolov8n.pt into the cwd; symlink the cached copy.
yolo_cache = f"{DRIVE}/models/yolov8n.pt"
if os.path.exists(yolo_cache) and not os.path.exists("yolov8n.pt"):
    os.symlink(yolo_cache, "yolov8n.pt")

print("caches wired to", DRIVE + "/models")

In [ ]:
# 6. FULL run. Repairs onnxruntime-gpu (forces the CUDA build to be
#    the SOLE onnxruntime, installed LAST so deps cannot shadow it),
#    then eluate -> occlude on the H.264 input. All speed fixes on:
#    1.2.0 = GPU image preprocess + GPU blur + YOLO fp16 + InsightFace
#    on CUDA + torch.compile ON + batched perception.
import os, shutil, subprocess, torch

assert torch.cuda.is_available(), "no CUDA - stop, fix runtime"
print("GPU OK:", torch.cuda.get_device_name(0), flush=True)

# Install tools, then force onnxruntime-gpu in last and alone. The
# uninstall+force-reinstall is REQUIRED: plain `onnxruntime` is an
# occlude core dep and shadows `onnxruntime-gpu`, so the [gpu] extra
# alone leaves InsightFace on CPU (cells 8/9 hit exactly this).
subprocess.run("pip install -q -U --index-url https://test.pypi.org/simple/ "
               "--extra-index-url https://pypi.org/simple/ "
               "'occlude[gpu]==1.2.0' eluate", shell=True, check=True)
subprocess.run("pip uninstall -y -q onnxruntime onnxruntime-gpu",
               shell=True, check=False)
subprocess.run("pip install -q --force-reinstall --no-deps onnxruntime-gpu",
               shell=True, check=True)

# Verify in a FRESH process - that is what the occlude subprocess sees
# (the kernel may have already imported the old CPU build). This loads
# all three models and reports the device EACH is bound to, so "all
# parts on GPU" is observed, not assumed. Hard-stops the run if the
# torch models or InsightFace failed to bind CUDA.
_verify = r'''
import onnxruntime, torch
from occlude.pipeline.perception import Perception
from occlude.pipeline import video as _v
from occlude.pipeline.io_cuda import cuda_io_available, cuda_io_unavailable_reasons
p = Perception(device="cuda")
seg = next(p.seg_model.parameters())
eps = set()
for m in p.face_app.models.values():
    s = getattr(m, "session", None)
    if s: eps |= set(s.get_providers())
yolo = getattr(p, "_yolo_device", "?")
print("ORT available  :", onnxruntime.get_available_providers())
print("torch device   :", p.device)
print("SegFormer      :", seg.device, seg.dtype)
print("GPU preprocess :", getattr(p, "_gpu_prep", "n/a"))
print("YOLO device    :", yolo, "| fp16:", getattr(p, "_yolo_half", "n/a"))
print("InsightFace EP :", sorted(eps))
print("Blur device    :", _v._BLUR_DEVICE)
print("CUDA video I/O :", cuda_io_available(), cuda_io_unavailable_reasons())
ok = (p.device.type == "cuda"
      and seg.device.type == "cuda"
      and getattr(p, "_gpu_prep", False) is True
      and yolo == 0
      and "CUDAExecutionProvider" in eps
      and _v._BLUR_DEVICE is not None)
print("ALL PARTS ON CUDA:", ok)
assert ok, "a part is NOT on CUDA - inspect the lines above"
'''
chk = subprocess.run(["python", "-c", _verify], capture_output=True, text=True)
print(chk.stdout.strip(), flush=True)
if chk.returncode != 0:
    print(chk.stderr.strip(), flush=True)
    raise SystemExit("GPU verification failed - do not start the full run")

DRIVE = "/content/drive/MyDrive/occlude"
IN  = f"{DRIVE}/inputs/The-Thinking-Game-h264.mp4"
OUT = f"{DRIVE}/outputs/The-Thinking-Game-occluded.mp4"
assert os.path.exists(IN), "input not on Drive - wait for upload"
assert os.path.getsize(IN) == 912805390, "input upload incomplete"
print("input verified", flush=True)
shutil.copy(IN, "/content/in.mp4")

# torch.compile is intentionally LEFT ON for the full run: the one-time
# ~10-30 frame inductor warm-up is negligible across a ~2 h movie and
# the fused SegFormer kernels are the single largest CUDA speedup. The
# short validation/profile cells (8, 9) disable it so their timings
# aren't dominated by the warm-up.

print(">>> STAGE 1: eluate (music removal)", flush=True)
!eluate /content/in.mp4 -o /content/in_nomusic.mp4 --device cuda

print(">>> STAGE 2: occlude (blur, batch 4) - hard-fails if CUDA EP does not bind",
      flush=True)
!python -m occlude --input /content/in_nomusic.mp4 --output /content/out.mp4 --device cuda --perception-batch 4

shutil.copy("/content/out.mp4", OUT)
print("DONE:", os.path.getsize("/content/out.mp4"), "bytes ->", OUT)


In [ ]:
# 7. (One-time, after the first run) cache YOLO weights to Drive so the
#    next session reuses them via the symlink in cell 5. Bandit v2, HF,
#    and InsightFace caches already persist automatically.
import shutil, os
if os.path.exists("yolov8n.pt") and not os.path.islink("yolov8n.pt"):
    shutil.copy("yolov8n.pt", f"{DRIVE}/models/yolov8n.pt")
    print("YOLO weights cached to Drive")

In [ ]:
# 8. VALIDATE occlude 1.2.0 (TestPyPI) on a 90s clip — GPU InsightFace.
#    Run this BEFORE trusting a full run. Self-contained; ~1 min.
import os, subprocess, time

# Same onnxruntime-gpu repair as cell 6 — REQUIRED. Plain `onnxruntime`
# is an occlude core dep; the [gpu] extra only ADDS onnxruntime-gpu, it
# can't remove the CPU build, and the CPU build shadows the GPU one.
# Installing the extra alone (the old cell 8) left InsightFace on CPU
# (~6 fps, the "WARNING: InsightFace is running on CPU" message). The
# uninstall-both + force-reinstall-gpu-last-and-alone is the fix.
subprocess.run("pip install -q -U --index-url https://test.pypi.org/simple/ "
               "--extra-index-url https://pypi.org/simple/ "
               "'occlude[gpu]==1.2.0'", shell=True, check=True)
subprocess.run("pip uninstall -y -q onnxruntime onnxruntime-gpu",
               shell=True, check=False)
subprocess.run("pip install -q --force-reinstall --no-deps onnxruntime-gpu",
               shell=True, check=True)

import importlib, occlude; importlib.reload(occlude)
print("occlude version:", occlude.__version__, flush=True)

DRIVE = "/content/drive/MyDrive/occlude"
SRC   = f"{DRIVE}/inputs/The-Thinking-Game-h264.mp4"

# 90s from a content-dense stretch (10:00-11:30).
subprocess.run(["ffmpeg","-y","-hide_banner","-loglevel","error",
                "-ss","600","-t","90","-i",SRC,
                "-c:v","libx264","-preset","veryfast","-crf","23",
                "-c:a","copy","/content/clip.mp4"], check=True)

# Compile OFF for this short clip only: the ~10-30 frame inductor
# warm-up would dominate a 90s timing and hide the real fps. The full
# run (cell 6) leaves compile ON.
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"]   = "1"

print(">>> occlude on 90s clip - hard-fails if any CUDA stage falls back", flush=True)
t0 = time.time()
!python -m occlude --input /content/clip.mp4 --output /content/clip_out.mp4 --device cuda --perception-batch 4
print(f">>> clip done in {time.time()-t0:.0f}s for 90s of video", flush=True)


In [ ]:
# 9. PROFILE occlude on a 30s clip - find the real bottleneck.
#    cProfile localizes the hot function. No guessing.
import os, subprocess, pstats, io

# Profile must reflect the GPU path, so do the same onnxruntime-gpu
# repair as cells 6/8 (the [gpu] extra alone leaves InsightFace on CPU
# and the profile would just show ONNX CPU inference dominating).
subprocess.run("pip install -q -U --index-url https://test.pypi.org/simple/ "
               "--extra-index-url https://pypi.org/simple/ "
               "'occlude[gpu]==1.2.0'", shell=True, check=True)
subprocess.run("pip uninstall -y -q onnxruntime onnxruntime-gpu",
               shell=True, check=False)
subprocess.run("pip install -q --force-reinstall --no-deps onnxruntime-gpu",
               shell=True, check=True)

DRIVE = "/content/drive/MyDrive/occlude"
SRC   = f"{DRIVE}/inputs/The-Thinking-Game-h264.mp4"

# short 30s clip keeps profiling fast.
subprocess.run(["ffmpeg","-y","-hide_banner","-loglevel","error",
                "-ss","600","-t","30","-i",SRC,
                "-c:v","libx264","-preset","veryfast","-crf","23",
                "-c:a","copy","/content/clip30.mp4"], check=True)

# Compile OFF for the 30s profile: with it on, the inductor warm-up
# would swamp a 30s sample and the profile would measure compilation,
# not steady-state. Full run (cell 6) keeps compile ON.
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"]   = "1"

print(">>> profiling occlude on 30s clip ...", flush=True)
subprocess.run(["python","-m","cProfile","-o","/content/occ.prof","-m","occlude",
                "--input","/content/clip30.mp4","--output","/content/clip30_out.mp4",
                "--device","cuda","--perception-batch","4"], check=True)

st = pstats.Stats("/content/occ.prof")
print("\n================ TOP 20 by SELF time (tottime) ================")
st.sort_stats("tottime").print_stats(20)
print("\n================ TOP 20 by CUMULATIVE time ================")
st.sort_stats("cumulative").print_stats(20)
